# Intro
There are three different e-commerce datasets from [Heywhale](https://www.heywhale.com/). So it is divided into three parts, with each part analyzing only one dataset.

## Dataset 1
`tmall_order_report.csv` contains order data. 

1. **Dimensions available for analysis：**
   - order time
   - location (收货地址)
2. **Metrics：**
   -  sales volume
   -  sales revenue
   -  refund amount
   -  return rate
   -  turnover rate
   -  regional distribution
   -  order time trends

### 1. Data Loading, exploration & pre-proccesing


In [37]:
import pandas as pd

df = pd.read_csv("./data/tmall_order_report.csv")

df.head()

,订单编号,总金额,买家实际支付金额,收货地址,订单创建时间,订单付款时间,退款金额
0,1,178.8,0.0,上海,2020-02-21 00:00:00,NaN,0.0
1,2,21.0,21.0,内蒙古自治区,2020-02-20 23:59:54,2020-02-21 00:00:02,0.0
2,3,37.0,0.0,安徽省,2020-02-20 23:59:35,NaN,0.0
3,4,157.0,157.0,湖南省,2020-02-20 23:58:34,2020-02-20 23:58:44,0.0
4,5,64.8,0.0,江苏省,2020-02-20 23:57:04,2020-02-20 23:57:11,64.8


In [38]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28010 entries, 0 to 28009
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   订单编号      28010 non-null  int64  
 1   总金额       28010 non-null  float64
 2   买家实际支付金额  28010 non-null  float64
 3   收货地址      28010 non-null  object 
 4   订单创建时间    28010 non-null  object 
 5   订单付款时间    24087 non-null  object 
 6   退款金额      28010 non-null  float64
dtypes: float64(3), int64(1), object(3)
memory usage: 1.5+ MB


In [39]:
print(df.columns, "\n")  # contain spaces in column names

print("=" * 50, "After handling", "=" * 50, "\n")
df.columns = df.columns.str.strip()
print(df.columns)

Index(['订单编号', '总金额', '买家实际支付金额', '收货地址 ', '订单创建时间', '订单付款时间 ', '退款金额'], dtype='object') 

================================================== After handling ================================================== 

Index(['订单编号', '总金额', '买家实际支付金额', '收货地址', '订单创建时间', '订单付款时间', '退款金额'], dtype='object')


In [40]:
df[df.duplicated()].count()  # no duplicated data

订单编号        0
总金额         0
买家实际支付金额    0
收货地址        0
订单创建时间      0
订单付款时间      0
退款金额        0
dtype: int64

In [41]:
df.isnull().sum()  # payment time has null values, indicating the order has not been paid for

订单编号           0
总金额            0
买家实际支付金额       0
收货地址           0
订单创建时间         0
订单付款时间      3923
退款金额           0
dtype: int64

## 2. Data Visualizing

### 2.1 Overall situation

In [42]:
result = {}

result["Total number of orders"] = df["订单编号"].count()
result["Total number of completed orders"] = df["订单编号"][df["订单付款时间"].notnull()].count()
result["Total number of uncomplete orders"] = df["订单编号"][df["订单付款时间"].isnull()].count()
result["Total number of refunded orders"] = df["订单编号"][df["退款金额"] > 0].count()
result["Total order amount"] = df["总金额"][df["订单付款时间"].notnull()].sum()
result["Total refunded order amount"] = df["退款金额"][df["订单付款时间"].notnull()].sum()
result["Total actual income"] = df["买家实际支付金额"][df["订单付款时间"].notnull()].sum()

In [43]:
result = pd.DataFrame(list(result.items()), columns=["Metrics", "Values"])

result

,Metrics,Values
0,Total number of orders,28010.00
1,Total number of completed orders,24087.00
2,Total number of uncomplete orders,3923.00
3,Total number of refunded orders,5646.00
4,Total order amount,2474823.07
5,Total refunded order amount,572335.92
6,Total actual income,1902487.15


In [44]:
refunded_orders = result[result["Metrics"] == "Total number of refunded orders"][
    "Values"
].iloc[0]
completed_orders = result[result["Metrics"] == "Total number of completed orders"][
    "Values"
].iloc[0]
total_orders = result[result["Metrics"] == "Total number of orders"]["Values"].iloc[0]

return_rate = refunded_orders / total_orders
turnover_rate = completed_orders / total_orders

print(f"Return Rate: {return_rate*100 :.2f}%")
print(f"Turnover Rate: {turnover_rate*100 :.2f}%")

Return Rate: 20.16%
Turnover Rate: 85.99%


### 2.2 Regional Analysis

In [45]:
from pyecharts.charts import Map, Line, Bar
from pyecharts import options as opts

regions_count = (
    df[df["订单付款时间"].notnull()]
    .groupby("收货地址")
    .agg({"订单编号": "count"})
    .to_dict()["订单编号"]
)

c = (
    Map()
    .add(
        series_name="订单量",
        data_pair=[*regions_count.items()],
        maptype="china",
        is_map_symbol_show=False,
    )
    .set_series_opts(label_opts=opts.LabelOpts(is_show=True))  # show region labels
    .set_global_opts(
        title_opts=opts.TitleOpts(title="地区分布"),  # figure title
        visualmap_opts=opts.VisualMapOpts(max_=1000),
    )
)

c.render_notebook()

### 2.3 Temporal Analysis

In [46]:
# convert to date format
df["订单创建时间"] = pd.to_datetime(df["订单创建时间"])
df["订单付款时间"] = pd.to_datetime(df["订单付款时间"])

In [47]:
date_count = (
    df.groupby(df["订单创建时间"].apply(lambda x: x.strftime("%d/%m/%Y")))
    .agg({"订单编号": "count"})
    .to_dict()["订单编号"]
)

c = (
    Line()
    .add_xaxis(list(date_count.keys()))
    .add_yaxis("订单量", list(date_count.values()))
    .set_series_opts(
        label_opts=opts.LabelOpts(is_show=False),
        markpoint_opts=opts.MarkPointOpts(
            data=[opts.MarkPointItem(type_="max", name="最大值")]
        ),
    )
    .set_global_opts(title_opts=opts.TitleOpts(title="每日订单量走势"))
)

c.render_notebook()

As shown in the chart above, the number of orders were **relatively low in the first half of February** due to the impact of COVID-19 pandemic. However, with the resumption of work, the number of orders **increased significantly in the second half of the month**.

In [48]:
hour_count = (
    df.groupby(df["订单创建时间"].apply(lambda x: x.strftime("%H")))
    .agg({"订单编号": "count"})
    .to_dict()["订单编号"]
)

c = (
    Bar()
    .add_xaxis(list(hour_count.keys()))
    .add_yaxis("订单量", list(hour_count.values()))
    .set_series_opts(
        label_opts=opts.LabelOpts(is_show=False),
        markpoint_opts=opts.MarkPointOpts(
            data=[opts.MarkPointItem(type_="max", name="最大值")]
        ),
    )
    .set_global_opts(title_opts=opts.TitleOpts(title="每小时订单量走势"))
)

c.render_notebook()

Looking at the hourly order volume trends, there are **three peak periods throughout the day(10:00, 15:00 and 21:00)**, with the **highest order volume occuring between 21:00 to 22:00**. To increase order volume, sellers should prioritize ensuring fast customer service responeses during peak periods, especially between 21:00 and 22:00. This is why many e-commerce business have night shifts.